# POC SOC Triage — 02 Feature Engineering

**Objectif** : construire les features ML à partir de `incidents_aggregated.csv` et des données simulées (TI, CMDB, Sandbox).

**Output** : `features_ml.csv` — 1 ligne par incident, prêt pour l'entraînement ML.

---
### Familles de features construites
| # | Famille | Nb features | Source |
|---|---------|-------------|--------|
| 1 | Alerte brute | 8 | incidents_aggregated |
| 2 | Temporelle | 6 | incidents_aggregated |
| 3 | Threat Intelligence | 6 | TI_database.csv |
| 4 | CMDB / Asset | 5 | CMDB.csv |
| 5 | Sandbox | 5 | Sandbox.csv |
| 6 | Historique SOC (taux FP) | 4 | incidents_aggregated |
| 7 | Contexte incident | 5 | incidents_aggregated |
| **Total** | | **~39** | |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Librairies OK')

## 1. Chargement des données

In [ ]:
df = pd.read_csv(DATA_DIR / 'incidents_aggregated.csv', parse_dates=['first_seen', 'last_seen'])
ti = pd.read_csv(DATA_DIR / 'TI_database.csv')
cmdb = pd.read_csv(DATA_DIR / 'CMDB.csv')
sandbox = pd.read_csv(DATA_DIR / 'Sandbox.csv')

print(f'Incidents : {len(df):,}')
print(f'Distribution cible :')
print(df['IncidentGrade'].value_counts())
print(f'\nTI IOCs : {len(ti):,}')
print(f'CMDB assets : {len(cmdb):,}')
print(f'Sandbox analyses : {len(sandbox):,}')

In [ ]:
# Aperçu rapide
df.head(3)

## 2. EDA rapide — distribution et valeurs manquantes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution de la cible
grade_counts = df['IncidentGrade'].value_counts()
colors = {'TP': '#E24B4A', 'BenignPositive': '#BA7517', 'FP': '#3B8BD4'}
bars = axes[0].bar(grade_counts.index, grade_counts.values,
                    color=[colors.get(g, 'gray') for g in grade_counts.index])
axes[0].set_title('Distribution IncidentGrade (cible)')
axes[0].set_ylabel('Nb incidents')
for bar, val in zip(bars, grade_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}\n({val/len(df):.1%})', ha='center', fontsize=9)

# Distribution du nombre d'alertes par incident
axes[1].hist(df['nb_alerts'].clip(upper=20), bins=20, color='#3B8BD4', edgecolor='white')
axes[1].set_title('Nb alertes par incident (cap 20)')
axes[1].set_xlabel('Nb alertes')
axes[1].set_ylabel('Nb incidents')

# Heure du jour
axes[2].hist(df['hour_of_day'], bins=24, color='#1D9E75', edgecolor='white')
axes[2].set_title('Heure du premier événement')
axes[2].set_xlabel('Heure')
axes[2].set_ylabel('Nb incidents')

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : eda_distributions.png')

In [ ]:
# Valeurs manquantes
missing = df.isnull().mean().sort_values(ascending=False)
missing_pct = missing[missing > 0] * 100
print('Colonnes avec valeurs manquantes (%) :')
print(missing_pct.round(1).to_string())

In [ ]:
# Taux de couverture des entités clés
entity_coverage = {
    'A une IP': df['has_ip'].mean(),
    'A un fichier': df['has_file'].mean(),
    'A un compte': df['has_account'].mean(),
    'A un device': df['has_device'].mean(),
    'A un email': df['has_email'].mean(),
    'A un hash': df['sample_sha256'].notna().mean(),
}
print('Couverture des entités par incident :')
for k, v in entity_coverage.items():
    print(f'  {k}: {v:.1%}')

## 3. Famille 1 — Features "Alerte brute"
Encodage des catégories, détecteurs, et titres d'alertes.

In [ ]:
feat = df[['IncidentId', 'OrgId', 'IncidentGrade']].copy()

# --- Taux de TP/FP/BP par AlertTitle (target encoding robuste) ---
# Calcul sur tout le dataset (pas de leakage car c'est une stat globale)
grade_dummies = pd.get_dummies(df['IncidentGrade'])
df_tmp = pd.concat([df[['IncidentId', 'top_alert_title', 'top_category',
                          'top_detector', 'nb_alerts', 'nb_evidences',
                          'nb_categories', 'nb_detectors', 'nb_entity_types']], grade_dummies], axis=1)

# Target encoding : proportion de TP/FP par AlertTitle
alert_title_stats = df_tmp.groupby('top_alert_title')[['TP', 'BenignPositive', 'FP']].mean()
alert_title_stats.columns = ['alert_title_tp_rate', 'alert_title_bp_rate', 'alert_title_fp_rate']
# Filtrer les titres trop rares (< 5 occurrences) → valeur globale
alert_title_counts = df_tmp['top_alert_title'].value_counts()
rare_titles = alert_title_counts[alert_title_counts < 5].index
global_tp_rate = df_tmp['TP'].mean()
global_fp_rate = df_tmp['FP'].mean()
global_bp_rate = df_tmp['BenignPositive'].mean()
alert_title_stats.loc[rare_titles, 'alert_title_tp_rate'] = global_tp_rate
alert_title_stats.loc[rare_titles, 'alert_title_fp_rate'] = global_fp_rate
alert_title_stats.loc[rare_titles, 'alert_title_bp_rate'] = global_bp_rate

# Target encoding : proportion de TP/FP par Category
cat_stats = df_tmp.groupby('top_category')[['TP', 'BenignPositive', 'FP']].mean()
cat_stats.columns = ['category_tp_rate', 'category_bp_rate', 'category_fp_rate']

# Merge
feat = feat.merge(df[['IncidentId', 'top_alert_title', 'top_category', 'top_detector',
                       'nb_alerts', 'nb_evidences', 'nb_categories',
                       'nb_detectors', 'nb_entity_types']], on='IncidentId')
feat = feat.merge(alert_title_stats, left_on='top_alert_title', right_index=True, how='left')
feat = feat.merge(cat_stats, left_on='top_category', right_index=True, how='left')

# Remplir les NaN
for col in ['alert_title_tp_rate', 'alert_title_fp_rate', 'alert_title_bp_rate',
            'category_tp_rate', 'category_fp_rate', 'category_bp_rate']:
    feat[col] = feat[col].fillna(feat[col].median())

print(f'Features alerte brute : {feat.shape[1] - 3} features')
feat[['alert_title_tp_rate', 'alert_title_fp_rate', 'category_tp_rate', 'category_fp_rate']].describe()

In [ ]:
# Vérification : les taux TP par AlertTitle sont-ils discriminants ?
fig, ax = plt.subplots(figsize=(10, 4))
for grade, color in [('TP', '#E24B4A'), ('BenignPositive', '#BA7517'), ('FP', '#3B8BD4')]:
    subset = feat[feat['IncidentGrade'] == grade]['alert_title_tp_rate']
    ax.hist(subset, bins=30, alpha=0.6, label=grade, color=color)
ax.set_xlabel('Taux TP du titre d\'alerte')
ax.set_ylabel('Nb incidents')
ax.set_title('Distribution du taux TP par AlertTitle — selon le grade réel')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_target_encoding.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Si les distributions sont bien séparées, la feature est discriminante ✓')

## 4. Famille 2 — Features temporelles

In [ ]:
feat['hour_of_day'] = df['hour_of_day']
feat['day_of_week'] = df['day_of_week']
feat['is_weekend'] = df['is_weekend']
feat['is_business_hours'] = df['is_business_hours']
feat['incident_duration_min'] = df['incident_duration_minutes'].clip(upper=10080)  # cap 1 semaine

# Feature : burst (nb alertes / durée) — indicateur de tempête d'alertes
feat['alert_rate_per_hour'] = np.where(
    df['incident_duration_minutes'] > 0,
    df['nb_alerts'] / (df['incident_duration_minutes'] / 60 + 0.01),
    df['nb_alerts']
).clip(upper=1000)

# Encodage cyclique de l'heure (pour capturer la continuité 23h → 0h)
feat['hour_sin'] = np.sin(2 * np.pi * feat['hour_of_day'] / 24)
feat['hour_cos'] = np.cos(2 * np.pi * feat['hour_of_day'] / 24)

print('Features temporelles ajoutées :')
print(['hour_of_day', 'day_of_week', 'is_weekend', 'is_business_hours',
       'incident_duration_min', 'alert_rate_per_hour', 'hour_sin', 'hour_cos'])

## 5. Famille 3 — Threat Intelligence
Jointure avec TI_database.csv sur l'IP et le hash de chaque incident.

In [ ]:
# Séparer TI par type
ti_ip = ti[ti['ioc_type'] == 'ip'][['ioc_value', 'ti_score', 'in_blocklist',
                                      'nb_sources_flagging', 'threat_categories']].copy()
ti_ip.columns = ['sample_ip', 'ip_ti_score', 'ip_in_blocklist',
                  'ip_nb_sources', 'ip_threat_categories']

ti_hash = ti[ti['ioc_type'] == 'sha256'][['ioc_value', 'ti_score', 'in_blocklist',
                                            'nb_sources_flagging']].copy()
ti_hash.columns = ['sample_sha256', 'hash_ti_score', 'hash_in_blocklist', 'hash_nb_sources']

# Jointure avec les incidents
feat = feat.merge(df[['IncidentId', 'sample_ip', 'sample_sha256']], on='IncidentId', how='left')
feat = feat.merge(ti_ip, on='sample_ip', how='left')
feat = feat.merge(ti_hash, on='sample_sha256', how='left')

# Valeurs par défaut si pas de TI disponible (= score neutre)
feat['ip_ti_score'] = feat['ip_ti_score'].fillna(0)
feat['ip_in_blocklist'] = feat['ip_in_blocklist'].fillna(0)
feat['ip_nb_sources'] = feat['ip_nb_sources'].fillna(0)
feat['hash_ti_score'] = feat['hash_ti_score'].fillna(0)
feat['hash_in_blocklist'] = feat['hash_in_blocklist'].fillna(0)
feat['hash_nb_sources'] = feat['hash_nb_sources'].fillna(0)

# Feature combinée : score TI max (IP ou hash)
feat['ti_score_max'] = feat[['ip_ti_score', 'hash_ti_score']].max(axis=1)
feat['any_ioc_in_blocklist'] = ((feat['ip_in_blocklist'] == 1) | (feat['hash_in_blocklist'] == 1)).astype(int)

# Taux de couverture TI
ip_coverage = feat['ip_ti_score'].gt(0).mean()
hash_coverage = feat['hash_ti_score'].gt(0).mean()
print(f'Couverture TI — IPs: {ip_coverage:.1%}, Hashes: {hash_coverage:.1%}')
print(f'Incidents avec IOC en blocklist: {feat["any_ioc_in_blocklist"].mean():.1%}')

# Vérification discriminante
print('\nScore TI max moyen par grade :')
print(feat.groupby('IncidentGrade')['ti_score_max'].mean().round(1))

## 6. Famille 4 — CMDB / Asset

In [ ]:
# Jointure CMDB sur le device name
cmdb_join = cmdb[['device_name', 'asset_criticality', 'sensitivity_score',
                    'is_internet_exposed', 'patch_level']].copy()
cmdb_join.columns = ['sample_device', 'asset_criticality', 'asset_sensitivity_score',
                      'asset_internet_exposed', 'asset_patch_level']

feat = feat.merge(df[['IncidentId', 'sample_device']], on='IncidentId', how='left')
feat = feat.merge(cmdb_join, on='sample_device', how='left')

# Encodage ordinal de la criticité
criticality_map = {'CRITICAL': 4, 'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
feat['asset_criticality_score'] = feat['asset_criticality'].map(criticality_map).fillna(2)
feat['asset_sensitivity_score'] = feat['asset_sensitivity_score'].fillna(40)
feat['asset_internet_exposed'] = feat['asset_internet_exposed'].fillna(0)

# Encodage patch level
patch_map = {'Critical-lag': 3, 'Minor-lag': 2, 'Up-to-date': 1}
feat['asset_patch_risk'] = feat['asset_patch_level'].map(patch_map).fillna(2)

print('CMDB features ajoutées')
print('\nCriticité asset moyenne par grade :')
print(feat.groupby('IncidentGrade')['asset_criticality_score'].mean().round(2))

## 7. Famille 5 — Sandbox

In [ ]:
sandbox_join = sandbox[['sha256', 'malware_score', 'network_connections',
                          'process_injections', 'c2_beaconing', 'sandbox_verdict']].copy()
sandbox_join.columns = ['sample_sha256', 'sandbox_malware_score', 'sandbox_network_conns',
                         'sandbox_process_injection', 'sandbox_c2_beaconing', 'sandbox_verdict']

feat = feat.merge(sandbox_join, on='sample_sha256', how='left')

# Valeurs par défaut : pas d'analyse = score neutre
feat['sandbox_malware_score'] = feat['sandbox_malware_score'].fillna(0)
feat['sandbox_network_conns'] = feat['sandbox_network_conns'].fillna(0)
feat['sandbox_process_injection'] = feat['sandbox_process_injection'].fillna(0)
feat['sandbox_c2_beaconing'] = feat['sandbox_c2_beaconing'].fillna(0)
feat['has_sandbox_analysis'] = feat['sandbox_verdict'].notna().astype(int)

print('Sandbox features ajoutées')
print(f'Incidents avec analyse sandbox: {feat["has_sandbox_analysis"].mean():.1%}')
print('\nScore sandbox moyen par grade :')
print(feat.groupby('IncidentGrade')['sandbox_malware_score'].mean().round(1))

## 8. Famille 6 — Historique SOC (taux FP par détecteur)

**C'est souvent la feature n°1 en importance.** Un détecteur qui génère 90% de FP dans l'historique génère très probablement un FP aujourd'hui.

> ⚠️ Attention au leakage : dans un contexte réel, on calculerait ce taux sur les 30 derniers jours *avant* l'alerte courante. Pour le POC, on utilise le taux global calculé sur tout le dataset. C'est à mentionner dans la présentation.

In [ ]:
grade_dummies2 = pd.get_dummies(df['IncidentGrade'])
df_for_stats = pd.concat([df[['IncidentId', 'top_detector', 'OrgId']], grade_dummies2], axis=1)

# Taux FP/TP par détecteur
detector_stats = df_for_stats.groupby('top_detector')[['TP', 'BenignPositive', 'FP']].agg(
    ['mean', 'count']
)
detector_stats.columns = ['_'.join(c) for c in detector_stats.columns]

# Garder seulement les détecteurs avec >= 10 incidents (sinon trop bruité)
detector_stats_filtered = detector_stats[detector_stats['TP_count'] >= 10].copy()
detector_stats_filtered = detector_stats_filtered.rename(columns={
    'TP_mean': 'detector_tp_rate',
    'FP_mean': 'detector_fp_rate',
    'BenignPositive_mean': 'detector_bp_rate',
    'TP_count': 'detector_nb_incidents',
})[['detector_tp_rate', 'detector_fp_rate', 'detector_bp_rate', 'detector_nb_incidents']]

feat = feat.merge(detector_stats_filtered, left_on='top_detector', right_index=True, how='left')

# Rares détecteurs → taux global
feat['detector_tp_rate'] = feat['detector_tp_rate'].fillna(df_for_stats['TP'].mean())
feat['detector_fp_rate'] = feat['detector_fp_rate'].fillna(df_for_stats['FP'].mean())
feat['detector_bp_rate'] = feat['detector_bp_rate'].fillna(df_for_stats['BenignPositive'].mean())
feat['detector_nb_incidents'] = feat['detector_nb_incidents'].fillna(0)

print('Features historique SOC :')
print('Taux FP moyen par détecteur (top 10 détecteurs les plus fréquents) :')
top_detectors = detector_stats_filtered.sort_values('detector_nb_incidents', ascending=False).head(10)
print(top_detectors[['detector_fp_rate', 'detector_tp_rate', 'detector_nb_incidents']].round(3))

## 9. Famille 7 — Contexte incident

In [ ]:
feat['has_ip'] = df['has_ip']
feat['has_file'] = df['has_file']
feat['has_account'] = df['has_account']
feat['has_email'] = df['has_email']

# Diversité des entités (plus c'est divers, plus c'est suspect)
feat['entity_diversity'] = df['nb_entity_types']

# Encodage de SuspicionLevel
suspicion_map = {'High': 3, 'Medium': 2, 'Low': 1, 'Unknown': 0}
feat['suspicion_level_encoded'] = df['max_suspicion'].map(suspicion_map).fillna(0)

# Présence d'une ThreatFamily connue
feat['has_threat_family'] = df['threat_family'].notna().astype(int)

# OS encodé (Windows plus ciblé)
feat['is_windows'] = (df['os_family'] == 'Windows').astype(int)

print('Features contexte incident ajoutées')
print(feat[['suspicion_level_encoded', 'entity_diversity', 'has_threat_family']].describe().round(2))

## 10. Sélection finale et encodage de la cible

In [ ]:
# Colonnes ML finales (sans les colonnes de jointure)
COLS_TO_DROP = [
    'OrgId', 'top_alert_title', 'top_category', 'top_detector',
    'sample_ip', 'sample_sha256', 'sample_device',
    'asset_criticality', 'asset_patch_level', 'sandbox_verdict',
    'ip_threat_categories',
]

feat_ml = feat.drop(columns=[c for c in COLS_TO_DROP if c in feat.columns])

# Encodage de la cible
GRADE_MAP = {'FP': 0, 'BenignPositive': 1, 'TP': 2}
feat_ml['target'] = feat_ml['IncidentGrade'].map(GRADE_MAP)
feat_ml = feat_ml.drop(columns=['IncidentGrade'])

# Vérification des NaN résiduels
null_check = feat_ml.isnull().sum()
null_remaining = null_check[null_check > 0]
if len(null_remaining) > 0:
    print('NaN résiduels à traiter :')
    print(null_remaining)
    # Remplissage par défaut
    feat_ml = feat_ml.fillna(0)
else:
    print('✓ Aucun NaN résiduel')

print(f'\nDataset ML final : {feat_ml.shape[0]:,} incidents × {feat_ml.shape[1]} colonnes')
print('\nDistribution cible :')
print(feat_ml['target'].value_counts().rename({0: 'FP', 1: 'BenignPositive', 2: 'TP'}))

In [ ]:
# Liste complète des features
feature_cols = [c for c in feat_ml.columns if c not in ['IncidentId', 'target']]
print(f'\n{len(feature_cols)} features ML :')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

## 11. Analyse de corrélation et importance préliminaire

In [ ]:
# Corrélation features vs cible
corr_with_target = feat_ml[feature_cols + ['target']].corr()['target'].drop('target').abs()
corr_with_target = corr_with_target.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = corr_with_target.head(20)
bars = ax.barh(range(len(top20)), top20.values, color='#3B8BD4')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Corrélation absolue avec la cible')
ax.set_title('Top 20 features — Corrélation avec IncidentGrade')
ax.axvline(0.1, color='red', linestyle='--', alpha=0.5, label='Seuil 0.10')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nTop 5 features les plus corrélées à la cible :')
print(corr_with_target.head(5).round(3))

In [ ]:
# Supprimer les features très peu corrélées (< 0.02) pour alléger le modèle
weak_features = corr_with_target[corr_with_target < 0.02].index.tolist()
if weak_features:
    print(f'Features faiblement corrélées (< 0.02), à surveiller : {weak_features}')
    # On les garde quand même — le ML peut trouver des interactions non linéaires
    print('  → Conservées pour le ML (XGBoost gère bien les features faibles)')
else:
    print('Toutes les features ont une corrélation >= 0.02 ✓')

## 12. Sauvegarde

In [ ]:
out_path = DATA_DIR / 'features_ml.csv'
feat_ml.to_csv(out_path, index=False)
print(f'✅ features_ml.csv sauvegardé : {feat_ml.shape[0]:,} incidents × {feat_ml.shape[1]} colonnes')
print(f'   Dont {len(feature_cols)} features ML + IncidentId + target')
print(f'\nRésumé des familles de features :')

families = {
    'Alerte brute (target encoding)': ['alert_title_tp_rate', 'alert_title_fp_rate', 'alert_title_bp_rate',
                                        'category_tp_rate', 'category_fp_rate', 'category_bp_rate',
                                        'nb_alerts', 'nb_evidences', 'nb_categories', 'nb_detectors'],
    'Temporelle': ['hour_of_day', 'day_of_week', 'is_weekend', 'is_business_hours',
                   'incident_duration_min', 'alert_rate_per_hour', 'hour_sin', 'hour_cos'],
    'Threat Intelligence': ['ip_ti_score', 'ip_in_blocklist', 'ip_nb_sources',
                             'hash_ti_score', 'hash_in_blocklist', 'hash_nb_sources',
                             'ti_score_max', 'any_ioc_in_blocklist'],
    'CMDB / Asset': ['asset_criticality_score', 'asset_sensitivity_score',
                      'asset_internet_exposed', 'asset_patch_risk'],
    'Sandbox': ['sandbox_malware_score', 'sandbox_network_conns',
                 'sandbox_process_injection', 'sandbox_c2_beaconing', 'has_sandbox_analysis'],
    'Historique SOC': ['detector_tp_rate', 'detector_fp_rate', 'detector_bp_rate', 'detector_nb_incidents'],
    'Contexte incident': ['has_ip', 'has_file', 'has_account', 'has_email',
                           'entity_diversity', 'suspicion_level_encoded',
                           'has_threat_family', 'is_windows', 'nb_entity_types'],
}

total = 0
for family, cols in families.items():
    present = [c for c in cols if c in feat_ml.columns]
    total += len(present)
    print(f'  {family}: {len(present)} features')
print(f'  TOTAL: {total} features')

print('\n→ Prochaine étape : 03_ml_models.ipynb')